# word2vec — word vectors from context (Skip-gram & CBOW)

> Tutorial pair for [`word2vec.py`](word2vec.py).

## 1. Intuition
"You shall know a word by the company it keeps." Words appearing in similar
contexts should get similar vectors. word2vec slides a window over text and
trains vectors so that a word predicts its neighbours (Skip-gram) — and the
resulting geometry encodes meaning (analogies become vector arithmetic).

## 2. Concept (the slide)
- **Skip-gram:** given the center word, predict each context word.
- **CBOW:** given the (averaged) context, predict the center word.
- Two embedding tables: "input" vectors $v_w$ and "output" vectors $u_w$.
- **Negative sampling** turns an expensive $|V|$-way softmax into a few cheap
  binary classifications.

## 3. Math derivation

**Full Skip-gram objective.** Maximize the probability of context words:
$$\frac1T\sum_t\sum_{-m\le j\le m,\,j\ne0}\log p(w_{t+j}\mid w_t),\qquad
  p(o\mid c)=\frac{\exp(u_o^\top v_c)}{\sum_{w\in V}\exp(u_w^\top v_c)}.$$
The denominator sums over the **whole vocabulary** — too expensive.

**Negative sampling (SGNS).** Replace it with: make the true pair score high and a
few random ("negative") pairs score low. For center $c$, true context $o$, and
negatives $k\sim P_n$:
$$\mathcal L=-\log\sigma(u_o^\top v_c)-\sum_{k}\mathbb E_{k\sim P_n}\log\sigma(-u_k^\top v_c).$$

**Gradients** (same clean form as logistic regression — let $s_w=\sigma(u_w^\top v_c)$):
$$\frac{\partial\mathcal L}{\partial u_o}=(s_o-1)v_c,\quad
  \frac{\partial\mathcal L}{\partial u_k}=(s_k-0)v_c,\quad
  \frac{\partial\mathcal L}{\partial v_c}=\sum_{w\in\{o\}\cup\text{neg}}(s_w-y_w)\,u_w.$$

**Noise distribution.** Negatives are drawn from the **unigram raised to the 3/4
power**, $P_n(w)\propto \text{count}(w)^{0.75}$ — empirically the sweet spot
between sampling frequent and rare words. Frequent words are also randomly
**subsampled** so "the" doesn't dominate.

**CBOW** is the mirror image: average the context vectors into $v_c=\frac1{|ctx|}\sum v_w$
and predict the center; the gradient w.r.t. each context vector is the averaged
$\partial\mathcal L/\partial v_c$.

## 4. NumPy implementation — Skip-gram/CBOW with negative sampling

In [ ]:
# ===== actual implementation from word2vec.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))

class Word2VecNumPy:
    r"""
    Two embedding tables: center vectors V (in) and context vectors U (out).
    Skip-gram + negative sampling objective (per (center c, true context o)):

        L = -log σ(u_o · v_c)  -  Σ_{k∈neg} log σ(-u_k · v_c)

    Gradients (clean, like logistic regression):
        for o:   du_o += (σ(u_o·v_c) - 1) v_c
        for k:   du_k += (σ(u_k·v_c) - 0) v_c
        dv_c    += Σ (σ(u_*·v_c) - label) u_*
    """

    def __init__(self, dim=50, window=2, neg=5, lr=0.05, mode="sg", seed=SEED):
        self.dim, self.window, self.neg = dim, window, neg
        self.lr, self.mode, self.seed = lr, mode, seed

    def build_vocab(self, sentences):
        from collections import Counter
        counts = Counter(w for s in sentences for w in s)
        self.itos = list(counts)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.counts = np.array([counts[w] for w in self.itos], float)
        # unigram^0.75 noise distribution for negative sampling
        p = self.counts ** 0.75
        self.noise = p / p.sum()
        V = len(self.itos)
        rng = np.random.default_rng(self.seed)
        self.V = (rng.random((V, self.dim)) - 0.5) / self.dim   # center
        self.U = np.zeros((V, self.dim))                        # context
        return self

    def _pairs(self, sentences):
        """Yield (center_idx, context_idx) within the window."""
        for s in sentences:
            ids = [self.stoi[w] for w in s if w in self.stoi]
            for i, c in enumerate(ids):
                lo, hi = max(0, i - self.window), min(len(ids), i + self.window + 1)
                ctx = [ids[j] for j in range(lo, hi) if j != i]
                if self.mode == "sg":
                    for o in ctx:
                        yield c, o
                else:  # CBOW: average context predicts the center
                    if ctx:
                        yield ctx, c

    def fit(self, sentences, epochs=50):
        self.build_vocab(sentences)
        rng = np.random.default_rng(self.seed)
        pairs = list(self._pairs(sentences))
        self.history = []
        for _ in range(epochs):
            rng.shuffle(pairs); loss = 0.0
            for a, b in pairs:
                if self.mode == "sg":
                    c, o = a, b
                    v = self.V[c]
                else:
                    ctx, o = a, b              # CBOW
                    v = self.V[ctx].mean(0)
                negs = rng.choice(len(self.itos), self.neg, p=self.noise)
                targets = np.concatenate([[o], negs])
                labels = np.concatenate([[1.0], np.zeros(self.neg)])
                scores = _sigmoid(self.U[targets] @ v)
                err = scores - labels                          # (1+neg,)
                dU = np.outer(err, v)
                dv = err @ self.U[targets]
                self.U[targets] -= self.lr * dU
                if self.mode == "sg":
                    self.V[c] -= self.lr * dv
                else:
                    self.V[ctx] -= self.lr * dv / len(ctx)
                loss += -np.log(scores[0] + 1e-9) - np.sum(np.log(1 - scores[1:] + 1e-9))
            self.history.append(loss / len(pairs))
        return self

    def vec(self, w): return self.V[self.stoi[w]]

    def most_similar(self, w, k=5):
        q = self.vec(w); q = q / (np.linalg.norm(q) + 1e-9)
        M = self.V / (np.linalg.norm(self.V, axis=1, keepdims=True) + 1e-9)
        sim = M @ q
        order = np.argsort(-sim)
        return [(str(self.itos[i]), float(sim[i])) for i in order if self.itos[i] != w][:k]

## 5. PyTorch implementation — SGNS with `nn.Embedding`

In [ ]:
# ===== actual implementation from word2vec.py =====
import torch

import torch.nn as nn

import torch.nn.functional as F

class SGNSTorch(nn.Module):
    def __init__(self, vocab, dim=50):
        super().__init__()
        self.center = nn.Embedding(vocab, dim)
        self.context = nn.Embedding(vocab, dim)
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.context.weight)

    def forward(self, c, pos, neg):
        v = self.center(c)                                  # (B, d)
        pos_s = (self.context(pos) * v).sum(-1)             # (B,)
        neg_s = torch.bmm(self.context(neg), v.unsqueeze(-1)).squeeze(-1)  # (B, K)
        loss = -(F.logsigmoid(pos_s) + F.logsigmoid(-neg_s).sum(1))
        return loss.mean()

def train_sgns_torch(w2v_np, sentences, dim=50, neg=5, epochs=50, lr=0.01):
    """Reuse the NumPy object's vocab/noise to train the torch version."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SGNSTorch(len(w2v_np.itos), dim).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(SEED)
    pairs = [(c, o) for c, o in w2v_np._pairs(sentences)]   # mode must be "sg"
    c = torch.tensor([p[0] for p in pairs], device=dev)
    o = torch.tensor([p[1] for p in pairs], device=dev)
    for _ in range(epochs):
        negs = torch.tensor(rng.choice(len(w2v_np.itos), (len(pairs), neg),
                                       p=w2v_np.noise), device=dev)
        opt.zero_grad(); loss = model(c, o, negs); loss.backward(); opt.step()
    return model

def toy_corpus():
    animals = "dog cat lion tiger horse cow".split()
    fruits = "apple banana orange grape mango pear".split()
    rng = np.random.default_rng(SEED)
    sents = []
    for _ in range(400):
        grp = animals if rng.random() < 0.5 else fruits
        sents.append(list(rng.choice(grp, size=5)))
    return sents

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    sents = toy_corpus()
    w = Word2VecNumPy(dim=20, window=2, neg=5, mode="sg").fit(sents, epochs=60)
    print("Skip-gram nearest neighbours:")
    for q in ("dog", "apple"):
        print(f"  {q:6s} -> {[t for t, _ in w.most_similar(q, 3)]}")

    cb = Word2VecNumPy(dim=20, mode="cbow").fit(sents, epochs=60)
    print(f"CBOW   'lion' -> {[t for t, _ in cb.most_similar('lion', 3)]}")

    train_sgns_torch(w, sents, dim=20, epochs=40)
    print("Torch SGNS trained (shares vocab & noise dist).")
    # sanity: an animal's neighbours should be animals, not fruits
    animals = set("dog cat lion tiger horse cow".split())
    nn3 = {t for t, _ in w.most_similar("dog", 3)}
    print(f"'dog' neighbours all animals: {nn3 <= animals}")

## 6. Train on a toy corpus — neighbours respect topic (animals vs fruits)

In [ ]:
demo()

## 7. Visualization — embeddings cluster by meaning (PCA to 2-D)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import word2vec as M

sents = M.toy_corpus()
w = M.Word2VecNumPy(dim=20, window=2, neg=5).fit(sents, epochs=60)
words = list(w.itos)
V = np.stack([w.vec(t) for t in words])
V = V - V.mean(0)
U, S, Vt = np.linalg.svd(V, full_matrices=False)   # PCA via SVD
P = V @ Vt[:2].T

animals = set("dog cat lion tiger horse cow".split())
plt.figure(figsize=(6, 5))
for (x, y), t in zip(P, words):
    color = "tab:red" if t in animals else "tab:green"
    plt.scatter(x, y, color=color); plt.annotate(str(t), (x, y), fontsize=9)
plt.title("word2vec embeddings (red=animals, green=fruits)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- SGNS = many small logistic regressions; gradient is the familiar $(\hat y-y)$.
- The $0.75$-power noise distribution and frequent-word subsampling matter in
  practice.
- Static embeddings give one vector per word — no sense disambiguation. That's
  what **contextual** models (ELMo → BERT) fix, using the
  [Transformer](../../05.transformers/architectures/transformer.ipynb).